[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/09-tokenization.ipynb)

# Module 9: Tokenization

Companion notebook to [Module 9](https://transformers-and-math.drmhse.com/modules/09-tokenization) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
from collections import Counter

## Part 1 — Byte-pair encoding from scratch (section 9.2)

In [ ]:
def learn_bpe(text, n_merges):
    words = Counter(text.lower().split())
    vocab = {tuple(w): c for w, c in words.items()}
    merges = []
    for _ in range(n_merges):
        pairs = Counter()
        for sym, c in vocab.items():
            for a, b in zip(sym, sym[1:]):
                pairs[a, b] += c
        if not pairs:
            break
        (a, b), count = pairs.most_common(1)[0]
        if count < 2:
            break
        merges.append((a, b))
        new = {}
        for sym, c in vocab.items():
            out, i = [], 0
            while i < len(sym):
                if i + 1 < len(sym) and (sym[i], sym[i + 1]) == (a, b):
                    out.append(a + b); i += 2
                else:
                    out.append(sym[i]); i += 1
            new[tuple(out)] = c
        vocab = new
    return merges

def tokenize(text, merges):
    tokens = []
    for word in text.lower().split():
        sym = list(word)
        for a, b in merges:
            i, out = 0, []
            while i < len(sym):
                if i + 1 < len(sym) and sym[i] == a and sym[i + 1] == b:
                    out.append(a + b); i += 2
                else:
                    out.append(sym[i]); i += 1
            sym = out
        tokens += sym
    return tokens

corpus = 'low ' * 5 + 'lower ' * 2 + 'newest ' * 6 + 'widest ' * 3
merges = learn_bpe(corpus, 8)
print('merges:', merges)
print(tokenize('the newest and lowest widget', merges))

## Part 2 — A real tokenizer (sections 9.2 and 9.5)
GPT-2's byte-level BPE via `tiktoken` (installed on Colab with pip if missing).

In [ ]:
try:
    import tiktoken
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tiktoken'], check=True)
    import tiktoken

try:
    enc = tiktoken.get_encoding('gpt2')
    for s in ['cat', ' cat', 'strawberry', '1234', '12345', 'café', 'antidisestablishmentarianism']:
        ids = enc.encode(s)
        print(f'{s!r:32} → {len(ids)} tokens: {[enc.decode([i]) for i in ids]}')
except Exception as e:      # tiktoken downloads its vocabulary on first use
    print('could not load the GPT-2 vocabulary (no network?):', e)

## Part 3 — Vocabulary size and cost (section 9.4)

In [ ]:
for V in (32_000, 50_257, 128_256, 256_000):
    print(f'|V| = {V:>7,}:  embedding table at d=4096 holds {V * 4096 / 1e6:,.0f}M parameters')

# Languages a tokenizer saw little of cost more tokens per character.
try:
    for s in ['The cat sat on the mat.', 'Le chat est assis sur le tapis.', 'Paka ameketi juu ya mkeka.']:
        print(f'{len(s) / len(enc.encode(s)):.2f} characters per token   {s}')
except NameError:
    pass

## Your turn
1. Run three BPE merges by hand on `aaab aab ab`, then check with `learn_bpe`.
2. How many tokens does GPT-2 use for a sentence in a language you know? Compare with English.